In [0]:
# Cell 1 — Config and inspect raw files
spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "03_mongodb"

dbutils.fs.ls(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/")

In [0]:
# Cell 2 — Read MongoDB products from S3 Raw and write to Bronze
df = spark.read.format("parquet") \
    .load(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=22/")

print(f"Raw count: {df.count()}")
df.printSchema()

In [0]:
# Cell 3 — Create schema and write to Bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.mongodb")

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.mongodb.products")

print("✅ bronze.mongodb.products written")
spark.sql("SELECT COUNT(*) as row_count FROM bronze.mongodb.products").show()